# Digitized Figure 2 reconstruction (superseded)

This notebook digitizes the mean and standard error points from the original paper's
Figure 2, before the authors sent raw data directly. **It is superseded**:
`00_raw_data_wrangle.ipynb` extracts the same aggregated values (and raw replicate
counts, which this digitization cannot recover) directly from the authors' Excel file,
and is what every downstream notebook actually uses. This notebook is kept for
provenance and as an independent cross-check on that extraction, not as a data source.

---

# 1. ETL: Extract-Transform-Load

The dataset used here was not obtained from raw experimental records. Instead, it was
reconstructed from published graphical summaries (Figure 2), where prey consumption
was reported as mean +/- standard error across four replicates per treatment
combination.

The ETL process therefore involves:

1. Digitizing plotted mean points and error bars.
2. Reconstructing per-treatment summary statistics:
   - Mean prey killed
   - Standard error (SE)
   - Sample size (n = 4)
3. Organizing the data into a tidy structure with the following variables:

   - `prey_density` (individuals mL-1)
   - `mean_consumed` (number of prey killed in 2 hours)
   - `se_consumed`
   - `salinity` (10, 20, 30 g L-1)
   - `prey_type` (`Apo`, `Nito`, `Bp` -- the same species codes used throughout the
     rest of this project, derived from each source file's name, not literal species
     names)

This notebook treats these summary-level observations as data generated from an
underlying experimental process. We do not use the fitted curves reported in the
original paper.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

In [ ]:
PROJ_DIR = Path.cwd().parent
DATA_DIR = PROJ_DIR / "data"
EXTR_DAT_DIR = DATA_DIR / "extracted_data"
SENT_DAT_DIR = DATA_DIR / "sent_data"
PROC_DATA_DIR = EXTR_DAT_DIR / "processed"      # THIS notebook's own output (d_final.csv)
SENT_PROC_DIR = SENT_DAT_DIR / "processed"       # 00_raw_data_wrangle.ipynb's output (hoja1_functional_response.csv)
data_files = list(EXTR_DAT_DIR.glob("figure_2/*.csv"))

In [ ]:
def extract_mean_se(group):
    # ASSUMES exactly 3 digitized (x, y) points per treatment-density group: the mean
    # marker and the two error-bar cap endpoints. If a source CSV has a different
    # number of digitized points for any group, this raises (too many/few values to
    # unpack) rather than silently mis-computing -- a loud failure, not a silent one,
    # but worth knowing if you extend the digitized point set.
    values = np.sort(group["prey_consumed"].values)
    low, mid, high = values
    mean = mid
    se = (high - low) / 2
    return pd.Series({
        "mean_consumed": mean,
        "se_consumed": se,
        "n": 4  # known from paper
    })


In [ ]:
def organize_data(file_):
    # Expects files named like <anything>_<salinity>_<prey_code>.csv, e.g.
    # "fig2_10_Apo.csv" -- salinity and prey type are read from the filename itself,
    # not from any column in the CSV.
    d = pd.read_csv(file_, header=None, names=["prey_density", "prey_consumed"], usecols=[0, 1])
    d['prey_density'] = d.prey_density.round()
    d['salinity'] = file_.stem.split("_")[1]
    d['prey_type'] = file_.stem.split("_")[2]
    return d

def process_stats(d):
    summary_df = (
        d
        .groupby(["prey_density", "salinity", "prey_type"])
        .apply(extract_mean_se, include_groups=False)
        .reset_index()
        )
    return summary_df


In [ ]:
d_final = pd.DataFrame()
for file in data_files:
    try:
        d_interim = process_stats(organize_data(file))
        d_final = pd.concat((d_final, d_interim))
    except Exception as e:
        print(e, file)

d_final = d_final.reset_index(drop=True)


In [ ]:
d_final.to_csv(PROC_DATA_DIR / "d_final.csv")

## Cross-check against the authoritative extraction

Compares this notebook's digitized reconstruction against `hoja1_functional_response.csv`
(the authors' actual data, extracted in `00_raw_data_wrangle.ipynb`) on `mean_consumed`
and `se_consumed`, joined on prey type, salinity, and offered density.

**This cross-check depends on `00_raw_data_wrangle.ipynb` having already been run** --
`hoja1_functional_response.csv` does not exist until that notebook produces it. Despite
the alphabetical filename order, this notebook must run second.

**Not independently tested here**: the join keys below assume `salinity` in this
notebook's digitized data is a bare numeric string (e.g. `"10"`) and that
`hoja1_functional_response.csv`'s `Salinity` column reads `"10 g/L"` -- adjust the
string formatting if the actual files differ.

In [ ]:
hoja1_path = SENT_PROC_DIR / "hoja1_functional_response.csv"
if not hoja1_path.exists():
    raise FileNotFoundError(
        f"{hoja1_path} not found -- run 00_raw_data_wrangle.ipynb first to produce it."
    )
d_sent = pd.read_csv(hoja1_path)

# Align join keys: digitized data's bare "10"/"Apo" vs. hoja1's "10 g/L"/"Apo"
d_final_j = d_final.copy()
d_final_j["Salinity"] = d_final_j["salinity"].astype(str) + " g/L"
d_final_j["Prey_Type"] = d_final_j["prey_type"]
d_final_j["N0"] = d_final_j["prey_density"]

merged = d_final_j.merge(
    d_sent, on=["Prey_Type", "Salinity", "N0"], suffixes=("_digitized", "_sent")
)
print(f"Matched {len(merged)} of {len(d_final_j)} digitized rows to the authoritative extract.")

if len(merged) > 0:
    mean_corr = merged["mean_consumed_digitized"].corr(merged["mean_consumed_sent"])
    mean_mae = (merged["mean_consumed_digitized"] - merged["mean_consumed_sent"]).abs().mean()
    se_corr = merged["se_consumed_digitized"].corr(merged["se_consumed_sent"])
    print(f"mean_consumed: r={mean_corr:.4f}, mean abs error={mean_mae:.3f}")
    print(f"se_consumed:   r={se_corr:.4f}")
else:
    print("No rows matched -- check the join-key formatting assumptions noted above.")
